# Module 7 Assignment: Rebuild Your Persona Agent on the OpenAI Agents SDK

Lab 6 rebuilt Study Buddy on top of the OpenAI Agents SDK: same persona, same notes, same three
tools from Lab 5, but with `Agent` and `Runner` replacing the hand written loop and dispatcher.

This assignment makes you do the same move on your **own** Persona Agent. By the end of Module 6
your agent had a persona, study notes, and three tools routed by a `call_tool(name, arguments)`
dispatcher you wrote by hand. Nothing about what your agent *does* needs to change here: you are
porting it to the framework, not redesigning it.

**What you'll build:**
1. An Azure configured `OpenAIChatCompletionsModel`, with tracing disabled, exactly like Lab 6
2. Your persona and notes, carried over from your Module 6 assignment
3. All three of your existing tools, each ported to `@function_tool`
4. One `Agent` that declares your persona, your notes, and all three tools
5. A test using `await Runner.run`
6. The same Gradio chat interface every lab and assignment has used since Lab 4, adapted from Lab 6

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 6 made the same kind of change to each of Study
Buddy's three tools, and do the analogous thing to yours.

## 0. Install the packages (run once)

Same as Lab 6.

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 1.00s
 Downloaded pydantic-core
 Downloaded openai
 Downloaded numpy
 Downloaded gradio
Prepared 10 packages in 4.92s
Uninstalled 10 packages in 5.03s
Installed 10 packages in 2.32s
 - filelock==4.0.12
 + filelock==4.1.1
 - gradio==6.29.1
 + gradio==6.30.0
 - numpy==2.5.3
 + numpy==2.5.4
 - openai==3.26.1
 + openai==3.29.0
 - pycparser==3.1
 + pycparser==3.11
 - pydantic==2.14.0
 + pydantic==2.14.1
 - pydantic-core==2.50.0
 + pydantic-core==2.50.1
 - pypdf==6.19.0
 + pypdf==6.20.1
 - starlette==1.7.0
 + starlette==1.8.0
 - truststore==0.10.4
 + truststore==0.10.5


## 1. Imports

**TODO:** Copy the imports from Lab 6, Step 1 (including `gradio as gr`), plus `requests` (and
anything else your own tools from the Module 6 assignment need, for example `html` if one of your
tools calls an API that returns HTML-escaped text).

In [2]:
# TODO: copy your imports from Lab 6 (including gradio as gr), plus requests and anything your tools need
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel, set_tracing_disabled
from pypdf import PdfReader
import gradio as gr

c:\Lab6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to Lab 6 and every assignment before it.

In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from Lab 6, Step 3, unchanged: call `set_tracing_disabled(True)`, build an
`AsyncAzureOpenAI` client from your endpoint, key, and API version, then wrap it and your deployment
name in `OpenAIChatCompletionsModel`.

In [4]:
# TODO: set_tracing_disabled(True), then build azure_client and azure_model, exactly like Lab 6
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint= AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. Your persona and notes

**TODO:** Bring over your persona description and your study notes from your Module 6 assignment,
unchanged. If your notes lived in a PDF, reuse the same `load_notes` helper from Lab 6, Step 4.

In [5]:
# TODO: persona = "..." and notes = "..." (or load_notes(...) if you used a PDF), from Module 6
persona = """You are a helpful study buddy agent with the following persona: NFL Study Budy.
Your goal is to help test people on their NFL knowledge using the provided notes."""

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

## 5. Port your first tool

**TODO:** Take your first tool from the Module 6 assignment and rewrite it as a plain, typed Python
function decorated with `@function_tool`, the same way Lab 6, Step 5a ported
`log_missed_question_tool`. Keep the function's actual behavior identical: only the decorator and
the removal of your hand written JSON schema should change.

In [6]:
# TODO: your first tool, decorated with @function_tool
@function_tool
def log_missed_question_tool(question: str) -> str:
    """Logs a question that the person missed to a file for later review.
    
    Args:
        question: The text or identifier of the missed question.
    """
    print(f"Tool called to log a missed question: {question}")
    with open("missed_questions.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

## 6. Port your second tool

**TODO:** Same idea, for your second tool (in Module 6 this was the one backed by a free, no-key
external API), the same way Lab 6, Step 5b ported `fetch_bonus_question_tool`.

In [7]:
# TODO: your second tool, decorated with @function_tool
@function_tool
def track_score_tool(correct: bool) -> str:
    """Updates and tracks the person's correct or incorrect answer counts.
    
    Args:
        correct: Indicates whether the person's answer was correct (true for correct, false for incorrect).
    """
    result_str = "CORRECT" if str(correct).lower() in ["true", "correct", "yes"] else "INCORRECT"
    
    print(f"Tool called to track score: {result_str}")
    
    with open("score_tracker.txt", "a", encoding="utf-8") as f:
        f.write(result_str + "\n")
        
    try:
        with open("score_tracker.txt", "r", encoding="utf-8") as f:
            lines = f.readlines()
            total_correct = sum(1 for line in lines if "CORRECT" in line)
            total_incorrect = sum(1 for line in lines if "INCORRECT" in line)
    except FileNotFoundError:
        total_correct = 1 if result_str == "CORRECT" else 0
        total_incorrect = 1 if result_str == "INCORRECT" else 0
        
    return f"Score updated! Total Correct: {total_correct}, Total Incorrect: {total_incorrect}"

## 7. Port your third tool

**TODO:** Same idea, for your third, self-designed tool, the same way Lab 6, Step 5c ported
`check_progress_tool`.

In [8]:
# TODO: your third tool, decorated with @function_tool
@function_tool
def hint_tool(question_id: Optional[str] = None) -> str:
    """Provides a subtle hint for the current NFL trivia or rules question when the person is stuck.
    
    Args:
        question_id: The unique identifier of the question (e.g., 'q1', 'q2', 'q3').
    """
    hint_database = {
        "q1": "Remember that the league consists of 32 teams split evenly between the American Football Conference (AFC) and the National Football Conference (NFC)[cite: 14].",
        "q2": "The offense is given four downs (attempts) to advance the ball a minimum of 10 yards to earn a new set of downs[cite: 14].",
        "q3": "Think back to the history of the AFL-NFL merger and early championship milestones, such as Super Bowl III won by the New York Jets."
    }
    
    if not question_id:
        return "Here is a general hint: Break the problem down into smaller steps and review your NFL study notes!"
        
    return hint_database.get(
        str(question_id).lower(), 
        "Break the problem down into smaller steps and review your notes for key NFL rules and history."
    )

## 8. Build your Agent

**TODO:** Build one `Agent`, the same way Lab 6, Step 6 built Study Buddy: a `name`, `instructions`
built from your persona and your notes (with the same rules your old system prompt had about when
to use each tool), a `tools` list containing all three of your ported tools, and `model=azure_model`.

Notice what is missing compared to your Module 6 assignment: no `call_tool(name, arguments)`
dispatcher, and no JSON schema for any of the three tools. `Agent` and `@function_tool` already
cover both.

In [9]:
# TODO: instructions = f"""..."""  (persona + notes + rules, like Module 6's system prompt)
# TODO: your_agent = Agent(name=..., instructions=instructions, tools=[...], model=azure_model)
instructions = f"""{persona}

You have access to three tools to assist the person:
1. Call `track_score_tool` whenever the person answers a question to update their correct or incorrect score.
2. Call `log_missed_question_tool` whenever the person answers a question incorrectly so it can be saved for later review.
3. Call `hint_tool` whenever the person explicitly asks for help or is struggling to answer a question.

Use the study notes below to answer questions:
{notes}
"""

your_agent = Agent(
    name="NFL Study Buddy",
    instructions=instructions,
    tools=[track_score_tool, log_missed_question_tool, hint_tool],
    model=azure_model,
)

print("Agent built successfully!")

Agent built successfully!


## 9. Test it: `await Runner.run`

**TODO:** Run your agent with `await Runner.run(your_agent, "...")`, the same way Lab 6, Step 7
tested Study Buddy. Pick a message that should trigger at least one of your three tools, and
confirm the tool's visible effect (a line appended to a file, a number that changed, or whatever
your tool does) actually happened.

Stick to `await Runner.run(...)` here, not `Runner.run_sync`: Jupyter notebooks already run their
own event loop in the background, and `Runner.run_sync` raises
`RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.` when
called from a notebook cell.

In [10]:
# TODO: result = await Runner.run(your_agent, "...")
# TODO: print(result.final_output)
result = await Runner.run(your_agent, input="Can you quiz me on NFL history?")
print(result.final_output)

Absolutely — let’s do an NFL history quiz.

I’ll ask one question at a time. You can also say “hint” if you want help.

**Question 1:**  
In what year was the NFL founded, originally as the American Professional Football Association?


## 10. Give it a real chat UI with Gradio

**TODO:** Adapt the `chat` function and `gr.ChatInterface` call from Lab 6, Step 8 to your own
agent: an `async def chat(message, history)` function that rebuilds `history` as a plain list of
`{"role": ..., "content": ...}` dictionaries, appends the new `message` as one more `user` turn,
passes that to `await Runner.run(your_agent, ...)`, and returns `result.final_output`, handed to
`gr.ChatInterface(chat).launch(inbrowser=True)`.

As in Lab 6, `gr.ChatInterface` accepts your async `chat` function directly, and its `history`
already comes in the shape `Runner.run` expects, so no extra conversion is needed.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [11]:
# TODO: async def chat(message, history): ... (see Lab 6, Step 8)
# TODO: gr.ChatInterface(chat).launch(inbrowser=True)
async def chat(message, history):
    history = [{"role": item["role"], "content": item["content"]} for item in history]
    result = await Runner.run(your_agent, history + [{"role": "user", "content": message}])
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. Which parts of your Module 6 assignment did you copy over completely unchanged? Which parts,
   if any, needed a real change rather than just a different decorator?
   ANSWER: I feel like most of it was the same as assignment 6 other than the runner and sdk
2. Compare your old `call_tool(name, arguments)` dispatcher to what `Agent` and `Runner` do for you
   now. What exactly is `Runner.run` doing in place of that dispatcher and your `while`/`for` loop?
   ANSWER: Instead of writing all the backend code like loops to check finish reasons, loops for tool calls, manual JSON parsing, and an if/elif dispatcher function—Runner.run does the entire process automatically. It manages the tool calls, runs the correct functions, and loops until the model finishes its response.
3. Your Module 6 system prompt included sentences telling the model when to use each tool. Did any
   of that wording need to change for the OpenAI Agents SDK, or did it carry over directly into
   `instructions`? ANSWER: I don't think any of the wording really changed at all because the model goes off of natural language along with the tools.
4. If you added a fourth tool to this agent, what would you need to write, and what would you not
   need to write, compared to adding a fourth tool by hand back in Module 6?
ANSWER: I'd need to write the new Python function with type hints, and the @function tool then put it into the tools list inside the agent. I wouldn't have to wtiyr any JSON schema or upadte the a different tool array for the API.
## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm all three of your ported tools are reachable, `tools=[...]` lists all three, and at least
  one tool call actually happened during your test in Step 9 or Step 10.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module7Assignment_JaneDoe.ipynb`, then submit
  it.